### Building LLM Agents with tool use

Locally implemented Python functions exposed to an LLM as **tools**.

**Task:** For profile `P001`, which genes connected to autism spectrum disorder should we prioritize, and why?

---

### Model backend: not decided yet — bring-your-own-key for now

> We have **not** finalised which model backend Session 3 will use (see
> `planning/session-3-framework-plan.md`). For now this notebook assumes a
> **bring-your-own-key** setup: it reads an `ANTHROPIC_API_KEY` from a local
> `.env` file and calls the real Anthropic API through LangChain.
>
> Per the framework plan, the tool-calling layer uses **LangChain** (the
> `@tool` decorator) wrapping the shared functions in `src/graph_tools.py`.
>
> **Setup:** create a `.env` file in the project root containing:
>
> ```
> ANTHROPIC_API_KEY=sk-ant-...
> ```
>
> The key is loaded into the environment and never printed. Run this notebook
> with the **`eccb`** conda environment / kernel.

## 0. Environment & API key

Load the `ANTHROPIC_API_KEY` from `.env`. We only check that it is present —
we never print it.

In [1]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# Load .env from the project root (does not overwrite already-set env vars).
load_dotenv(PROJECT_ROOT / ".env")

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY not found. Create a .env file in the project root "
        "with `ANTHROPIC_API_KEY=sk-ant-...` (bring-your-own-key)."
    )

print("ANTHROPIC_API_KEY loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

ANTHROPIC_API_KEY loaded: True


## 1. Load the shared graph tools

These functions in `src/graph_tools.py` are the shared backend for the tools,
MCP, and skill tracks. (Currently toy implementations — confirm the real tool
set with the Session 1 & 2 groups.)

In [2]:
from src.graph_tools import (
    build_toy_graph,
    get_neighbors,
    rank_candidate_genes,
    rank_profile_features,
    search_nodes,
)

graph = build_toy_graph()
graph

## 2. Demonstrate Tool Outputs

In [3]:
phenotype_matches = search_nodes(graph, "autism")
phenotype_matches

[{'node_id': 'HP:0000729',
  'name': 'Autism spectrum disorder',
  'type': 'phenotype'}]

In [4]:
phenotype_id = phenotype_matches[0]["node_id"]
gene_neighbors = get_neighbors(graph, phenotype_id, node_type="gene")
gene_neighbors

[{'node_id': 'GENE:SHANK3',
  'direction': 'incoming',
  'edge_to': 'HP:0000729',
  'node': {'name': 'SHANK3', 'type': 'gene'},
  'edge': {'relation': 'associated_with', 'evidence': 'toy literature edge'}},
 {'node_id': 'GENE:CHD8',
  'direction': 'incoming',
  'edge_to': 'HP:0000729',
  'node': {'name': 'CHD8', 'type': 'gene'},
  'edge': {'relation': 'associated_with', 'evidence': 'toy literature edge'}},
 {'node_id': 'GENE:SCN2A',
  'direction': 'incoming',
  'edge_to': 'HP:0000729',
  'node': {'name': 'SCN2A', 'type': 'gene'},
  'edge': {'relation': 'associated_with', 'evidence': 'toy literature edge'}}]

In [5]:
profile_features = rank_profile_features("P001")
profile_features

[{'feature_id': 'GENE:SHANK3', 'score': 2.4},
 {'feature_id': 'GENE:SCN2A', 'score': 1.7},
 {'feature_id': 'GENE:CHD8', 'score': 0.8}]

In [6]:
ranked_candidates = rank_candidate_genes(graph, "P001", phenotype_id)
ranked_candidates

[{'gene_id': 'GENE:SHANK3',
  'name': 'SHANK3',
  'profile_score': 2.4,
  'phenotype_id': 'HP:0000729'},
 {'gene_id': 'GENE:SCN2A',
  'name': 'SCN2A',
  'profile_score': 1.7,
  'phenotype_id': 'HP:0000729'},
 {'gene_id': 'GENE:CHD8',
  'name': 'CHD8',
  'profile_score': 0.8,
  'phenotype_id': 'HP:0000729'}]

## 3. Wrap the functions as LangChain tools

The `@tool` decorator turns a plain Python function into a tool the model can
call. The **docstring becomes the tool description** the model reads. Each wrapper closes over the `graph` object built above.

In [7]:
from langchain_core.tools import tool


@tool
def search_nodes_tool(query: str) -> list:
    """Search for nodes in the biomedical knowledge graph by name or ID substring.

    Use this to resolve a free-text term (e.g. 'autism') to a node id such as a
    phenotype or gene identifier.
    """
    return search_nodes(graph, query)


@tool
def get_neighbors_tool(node_id: str, node_type: str | None = None) -> list:
    """Get neighboring nodes for a node id, with edge metadata.

    Optionally filter neighbors by node_type (e.g. 'gene', 'phenotype',
    'pathway').
    """
    return get_neighbors(graph, node_id, node_type=node_type)


@tool
def rank_profile_features_tool(profile_id: str, top_n: int = 20) -> list:
    """Rank the top molecular features for a multi-omic profile by score."""
    return rank_profile_features(profile_id, top_n=top_n)


@tool
def rank_candidate_genes_tool(profile_id: str, phenotype_id: str) -> list:
    """Rank genes connected to a phenotype, scored by the profile's feature data.

    Returns genes that are both linked to phenotype_id in the graph and present
    in the profile, sorted by profile score (highest first).
    """
    return rank_candidate_genes(graph, profile_id, phenotype_id)


tools = [
    search_nodes_tool,
    get_neighbors_tool,
    rank_profile_features_tool,
    rank_candidate_genes_tool,
]
tools_by_name = {t.name: t for t in tools}
tools_by_name

{'search_nodes_tool': StructuredTool(name='search_nodes_tool', description="Search for nodes in the biomedical knowledge graph by name or ID substring.\n\nUse this to resolve a free-text term (e.g. 'autism') to a node id such as a\nphenotype or gene identifier.", args_schema=<class 'langchain_core.utils.pydantic.search_nodes_tool'>, func=<function search_nodes_tool at 0x10757e480>),
 'get_neighbors_tool': StructuredTool(name='get_neighbors_tool', description="Get neighboring nodes for a node id, with edge metadata.\n\nOptionally filter neighbors by node_type (e.g. 'gene', 'phenotype',\n'pathway').", args_schema=<class 'langchain_core.utils.pydantic.get_neighbors_tool'>, func=<function get_neighbors_tool at 0x1077d1ee0>),
 'rank_profile_features_tool': StructuredTool(name='rank_profile_features_tool', description='Rank the top molecular features for a multi-omic profile by score.', args_schema=<class 'langchain_core.utils.pydantic.rank_profile_features_tool'>, func=<function rank_profil

## 4. Connect the tools to a Claude model

We bind the tools to a Claude chat model via LangChain. `bind_tools` tells the
model which tools exist and their schemas; the model decides *when* to call
them.

In [ ]:
from langchain_anthropic import ChatAnthropic

# Model backend is not finalised (see planning/session-3-framework-plan.md).
MODEL = "claude-sonnet-4-6"

llm = ChatAnthropic(model=MODEL, temperature=0)
llm_with_tools = llm.bind_tools(tools)
print("Using model:", MODEL)

Using model: claude-sonnet-4-6


## 5. Run the agent loop

A tool-calling agent is just a loop:

1. Send the conversation to the model.
2. If the model asks to call tools, run them and append the results.
3. Repeat until the model returns a final answer with no more tool calls.

The `verbose` flag prints each tool call so participants can see the evidence
the model gathered.

In [9]:
import json

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage


def run_agent(question: str, max_steps: int = 6, verbose: bool = True) -> AIMessage:
    """Minimal tool-calling loop over the bound Claude model."""
    messages = [
        SystemMessage(
            content=(
                "You are a biomedical research assistant. Use the available "
                "tools to query the knowledge graph and profile data. Always "
                "ground factual claims in tool results, and briefly cite which "
                "tool evidence supports your answer."
            )
        ),
        HumanMessage(content=question),
    ]

    for step in range(max_steps):
        ai_msg = llm_with_tools.invoke(messages)
        messages.append(ai_msg)

        if not ai_msg.tool_calls:
            return ai_msg

        for call in ai_msg.tool_calls:
            if verbose:
                print(f"[step {step}] -> {call['name']}({call['args']})")
            result = tools_by_name[call["name"]].invoke(call["args"])
            messages.append(
                ToolMessage(
                    content=json.dumps(result, default=str),
                    tool_call_id=call["id"],
                )
            )

    raise RuntimeError(f"Agent did not finish within {max_steps} steps.")

In [10]:
question = (
    "For profile P001, which genes connected to autism spectrum disorder "
    "should we prioritize, and why?"
)

final = run_agent(question)
print("\n=== FINAL ANSWER ===\n")
print(final.content)

[step 0] -> search_nodes_tool({'query': 'autism spectrum disorder'})
[step 0] -> rank_profile_features_tool({'profile_id': 'P001'})
[step 1] -> rank_candidate_genes_tool({'profile_id': 'P001', 'phenotype_id': 'HP:0000729'})

=== FINAL ANSWER ===

Excellent! Here is a comprehensive prioritization of genes connected to **Autism Spectrum Disorder (HP:0000729)** for **Profile P001**, ranked by their profile scores:

---

## 🧬 Gene Prioritization for Autism Spectrum Disorder — Profile P001

| Rank | Gene | Profile Score | Priority |
|------|--------|--------------|----------|
| 1 | **SHANK3** | 2.4 | 🔴 High |
| 2 | **SCN2A** | 1.7 | 🟠 Medium-High |
| 3 | **CHD8** | 0.8 | 🟡 Medium |

---

### 🔴 1. SHANK3 — Score: 2.4 (Top Priority)
**SHANK3** is the highest-scoring gene in both the overall profile feature ranking and the ASD-specific candidate ranking. SHANK3 encodes a scaffolding protein critical for postsynaptic density organization at excitatory synapses. Mutations in SHANK3 are strongly 

## 6. Compare against the direct tool evidence

The model's answer should match the ranking we computed by hand in Section 2 —
this is the ground truth the agent was expected to reproduce (`SHANK3`,
`SCN2A`, `CHD8`).

In [11]:
print("Expected ranking (computed directly):")
for item in ranked_candidates:
    print(
        f"  {item['name']}: profile score {item['profile_score']} "
        f"(linked to {item['phenotype_id']})"
    )

Expected ranking (computed directly):
  SHANK3: profile score 2.4 (linked to HP:0000729)
  SCN2A: profile score 1.7 (linked to HP:0000729)
  CHD8: profile score 0.8 (linked to HP:0000729)


## Reflection

- Which facts in the model's answer came from the **graph** vs. the **profile**?
- Which tool calls did the model choose, and in what order?
- What would be unsafe for a model to infer **without** calling these tools?
- How does this compare to the MCP track (`02_MCP.ipynb`) and the
  skills track (`03_agent_skills.ipynb`), which reuse the same
  `graph_tools.py` backend?